# Import

In [1]:
import math
import torch
import torch.nn as nn

In [2]:
class MultiHeadAttention(nn.Module):
    
    def __init__(self, d_model, num_heads):
        super().__init__()
        
        # assert is keyword
        # its a debugging tools used to test if a specific condition in our code evalutes to true
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"

        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads


        self.Wq = nn.Linear(in_features = d_model, out_features = d_model)
        self.Wk = nn.Linear(in_features = d_model, out_features = d_model)
        self.Wv = nn.Linear(in_features = d_model, out_features = d_model)

        self.out_proj = nn.Linear(in_features = d_model, out_features = d_model)

    def forward(self, x):

        batch_size, seq_len, d_model = x.shape

        print('=' * 10)
        print("x shape ", x.shape)

        #send x in linear layer

        Q = self.Wq(x)
        K = self.Wk(x)
        V = self.Wv(x)

        print("\n After Linear Layers")

        print("Q : ", Q.shape)
        print("K : ", K.shape)
        print("V : ", V.shape)

        # split into heads using view
        # view is like reshaping but without copy 

        Q = Q.view(
            batch_size,
            seq_len,
            self.num_heads,
            self.head_dim
        )

        K = K.view(
            batch_size,
            seq_len,
            self.num_heads,
            self.head_dim
        )

        V = V.view(
            batch_size,
            seq_len,
            self.num_heads,
            self.head_dim
        )
        

        print("\n After reshaping")
        print(Q.shape)
        print(K.shape)
        print(V.shape)

        # Move heads forward
        
        # change shape 
        Q = Q.transpose(1, 2)
        K = K.transpose(1, 2)
        V = V.transpose(1, 2)

        print("\n After Transpose")
        print("Q : ", Q.shape)
        print("K : ", K.shape)
        print("V : ", V.shape)

        # Attention score

        scores = Q @ K.transpose(-2, -1) / math.sqrt(self.head_dim)

        print("\nscore")
        print(scores.shape)

        # softmax

        weights = torch.softmax(scores, dim = -1)

        print("\nAfter softmax Weight Shape")
        print(weights.shape)

        print("\nHead 1 Attentation matrix Weights")
        print(weights[0, 0])
        print("\nHead 2 Attentation matrix Weights")
        print(weights[0, 1])
        
        # Apply Attention
        output = weights @ V

        print("\nAfter Weights @ V")
        print(output.shape)

        # Combine heads
        output = output.transpose(1, 2)

        print("\nAfter Transpose Back")
        print(output.shape)

        # After Transpose we can't use view to reshape it 
        # so contiguous allocates a new block of memory to copy and rearrange a tensor's data into a sequential, unbroken memory layout.
        # then apply view()

        # print(output.contiguous())

        output = output.contiguous().view(
            batch_size,
            seq_len,
            d_model
        )

        # After Flatten
        print("\nAfter Flatten heads (2, 4 to 8)")
        print(output.shape)

        # Final projection

        output = self.out_proj(output)

        print("\nFinal output")
        print(output.shape)

        return output 

In [3]:
# Example input

batch_size = 1
seq_len = 3
d_model = 8
num_heads = 2

vocab = {
    "I" : 0,
    "Love" : 1,
    "AI" : 2
}

tokens = torch.tensor([0, 1, 2]) # or torch.tensor(vocab.values())

embedding = nn.Embedding(
    num_embeddings = len(vocab),
    embedding_dim = d_model
)

x  = embedding(tokens)

x = x.view(1, 3, 8)

print(x.shape)
print(x)

mha = MultiHeadAttention(d_model = d_model, num_heads = num_heads)

output = mha(x)

print("\noutput")
# print(output.shape)

torch.Size([1, 3, 8])
tensor([[[-1.3792, -0.4835, -0.8596,  0.7727,  1.5673,  0.1496,  1.1794,
          -0.6461],
         [-0.7051,  0.2597, -1.8463, -0.1359, -1.7742, -1.1541, -0.2551,
          -1.0613],
         [-1.4461, -1.0041, -0.3395,  1.0609,  0.3884, -0.4831,  2.3642,
          -0.0274]]], grad_fn=<ViewBackward0>)
x shape  torch.Size([1, 3, 8])

 After Linear Layers
Q :  torch.Size([1, 3, 8])
K :  torch.Size([1, 3, 8])
V :  torch.Size([1, 3, 8])

 After reshaping
torch.Size([1, 3, 2, 4])
torch.Size([1, 3, 2, 4])
torch.Size([1, 3, 2, 4])

 After Transpose
Q :  torch.Size([1, 2, 3, 4])
K :  torch.Size([1, 2, 3, 4])
V :  torch.Size([1, 2, 3, 4])

score
torch.Size([1, 2, 3, 3])

After softmax Weight Shape
torch.Size([1, 2, 3, 3])

Head 1 Attentation matrix Weights
tensor([[0.2666, 0.4965, 0.2369],
        [0.3221, 0.3628, 0.3150],
        [0.2962, 0.4333, 0.2705]], grad_fn=<SelectBackward0>)

Head 2 Attentation matrix Weights
tensor([[0.3600, 0.4198, 0.2202],
        [0.2105, 0